# EDA — criminalite.csv
Score de Vivabilité · Statistiques de criminalité par commune (CODGEO)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130

## 1. Chargement

In [ ]:
FILE = 'architecture-data/brute/score_de_vivabilité/criminalite.csv'

df = pd.read_csv(FILE, sep=None, engine='python')
df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=False)

print(f'Shape : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
df.head()

## 2. Infos générales

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 3. Valeurs manquantes — critique pour ce dataset

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, max(3, len(df.columns) * 0.45)))
colors = ['#D85A30' if v > 50 else '#BA7517' if v > 20 else '#1D9E75' for v in missing.values]
ax.barh(missing.index, missing.values, color=colors)
ax.set_xlabel('% manquant')
ax.set_title('Valeurs manquantes par colonne (rouge > 50%, orange > 20%)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.axvline(50, color='#D85A30', linestyle='--', linewidth=0.8, alpha=0.6)
plt.tight_layout()
plt.show()

display(missing.rename('% manquant').to_frame())

## 4. Doublons

In [ ]:
n_dup = df.duplicated().sum()
print(f'Doublons : {n_dup} ({n_dup/len(df)*100:.2f}%)')
if n_dup > 0:
    display(df[df.duplicated(keep=False)].head(10))

## 5. Détection colonnes clés

In [ ]:
def find_col(df, *keywords):
    for kw in keywords:
        match = [c for c in df.columns if kw.lower() in c.lower()]
        if match: return match[0]
    return None

col_map = {
    'codgeo'      : find_col(df, 'codgeo', 'insee', 'code_commune'),
    'annee'       : find_col(df, 'annee', 'année', 'year'),
    'indicateur'  : find_col(df, 'indicateur', 'type', 'categorie'),
    'nombre'      : find_col(df, 'nombre', 'count', 'nb'),
    'taux'        : find_col(df, 'taux', 'rate', 'pour_mille'),
    'diffuse'     : find_col(df, 'diffuse', 'diffusé', 'est_diff'),
    'unite'       : find_col(df, 'unite', 'unité', 'unit'),
}

print('Colonnes détectées :')
for k, v in col_map.items():
    status = '✅' if v else '❌'
    print(f'  {status}  {k:20s} → {v}')
print(f'\nColonnes non mappées : {[c for c in df.columns if c not in col_map.values()]}')

## 6. Données diffusées vs non diffusées

In [ ]:
diff_col = col_map['diffuse']
if diff_col:
    vc = df[diff_col].value_counts()
    print(vc.to_string())

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.pie(vc.values, labels=vc.index, autopct='%1.1f%%',
           colors=['#1D9E75', '#D85A30'], startangle=90)
    ax.set_title('Part diffusée vs non diffusée')
    plt.tight_layout()
    plt.show()
else:
    print('❌ Colonne diffuse non trouvée')

## 7. Répartition par type d'indicateur

In [ ]:
ind_col = col_map['indicateur']
if ind_col:
    vc_ind = df[ind_col].value_counts()
    fig, ax = plt.subplots(figsize=(10, max(4, len(vc_ind) * 0.4)))
    ax.barh(vc_ind.index, vc_ind.values, color='#378ADD')
    ax.set_title("Nombre de lignes par type d'indicateur de criminalité")
    ax.set_xlabel('Nombre de lignes')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    display(vc_ind.rename('count').to_frame())

## 8. Distribution du nombre de faits (données diffusées)

In [ ]:
nombre_col = col_map['nombre']
diff_col   = col_map['diffuse']

df_diff = df[df[diff_col] == 'diff'].copy() if diff_col else df.copy()
print(f'Lignes diffusées : {len(df_diff):,}')

if nombre_col and df_diff[nombre_col].notna().sum() > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(df_diff[nombre_col].dropna(), bins=40, color='#378ADD', edgecolor='white')
    axes[0].set_title('Distribution du nombre de faits')
    axes[0].set_xlabel('Nombre de faits')

    axes[1].hist(np.log1p(df_diff[nombre_col].dropna()), bins=40, color='#1D9E75', edgecolor='white')
    axes[1].set_title('Distribution log1p (nombre de faits)')
    axes[1].set_xlabel('log1p(nombre)')

    plt.suptitle('Distribution des faits de criminalité', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. Top communes par volume de faits

In [ ]:
codgeo_col = col_map['codgeo']
nombre_col = col_map['nombre']

if codgeo_col and nombre_col:
    top_communes = (
        df_diff.groupby(codgeo_col)[nombre_col]
        .sum()
        .sort_values(ascending=False)
        .head(20)
    )
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(top_communes.index.astype(str), top_communes.values, color='#D85A30')
    ax.set_title('Top 20 communes — volume total de faits (toutes catégories)')
    ax.set_xlabel('Nombre total de faits')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 10. Résumé + plan Silver

In [ ]:
print('=' * 55)
print('RÉSUMÉ EDA — criminalite.csv')
print('=' * 55)
print(f'  Lignes              : {len(df):,}')
print(f'  Colonnes            : {len(df.columns)}')
print(f'  Doublons            : {df.duplicated().sum()}')
if col_map['codgeo']:   print(f'  Communes (CODGEO)   : {df[col_map["codgeo"]].nunique()}')
if col_map['annee']:    print(f'  Année(s)            : {df[col_map["annee"]].unique()}')
if col_map['indicateur']: print(f'  Indicateurs         : {df[col_map["indicateur"]].nunique()}')
if diff_col:            print(f'  Lignes diffusées    : {(df[diff_col]=="diff").sum()} ({(df[diff_col]=="diff").mean()*100:.1f}%)')
print('=' * 55)
print()
print('ACTIONS SILVER REQUISES :')
print('  → Filtrer est_diffuse == "diff" avant tout calcul')
print('  → Convertir taux_pour_mille (str avec virgule) en float')
print('  → Pivoter indicateur → colonnes pour 1 ligne par CODGEO')
print('  → Utiliser taux_pour_mille (normalisé par pop) pour le score Gold')
print('  → ⚠️  Snapshot 2016 uniquement — donnée ancienne à signaler')